In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# ✅ Only keep this import
from langchain_community.chat_models import ChatOllama

# -----------------------------
# 1. Load dataset
# -----------------------------
df = pd.read_csv('cleaned1012_dataset.csv')

# -----------------------------
# 2. Target columns
# -----------------------------
target_columns = [
    'Inflation_YoY',
    'Oil_Price_USD_Barrel',
    'Exchange_Rate_PKR_USD',
    'Interest_Rate',
    'Money_Supply_M2_Billion',
    'pkr_to_oneDollar',
    'Gold_Price_In_Dolars',
    'Petrol_Price',
    'Minimum_Wage_PKR'
]

# -----------------------------
# 3. Train models
# -----------------------------
models = {}

existing_targets = [col for col in target_columns if col in df.columns]

for col in existing_targets:
    valid_data = df.dropna(subset=[col, 'Year'])

    X_train = valid_data[['Year']].values
    y_train = valid_data[col].values

    model = LinearRegression()
    model.fit(X_train, y_train)

    models[col] = model


# -----------------------------
# 4. LLM Setup (Ollama)
# -----------------------------
llm = ChatOllama(model="llama3:latest")


# -----------------------------
# 5. Prediction Function
# -----------------------------
def predict_economy(target_year, explain=False):
    year_input = np.array([[target_year]])
    predictions = {}

    print(f"\n--- Economic Forecast for {target_year} ---")
    print("-" * 50)

    for name, model in models.items():
        pred = model.predict(year_input)[0]
        predictions[name] = round(pred, 2)

        label = name.replace('_', ' ').title()
        print(f"{label:<30}: {predictions[name]}")

    # Highlight Minimum Wage
    if "Minimum_Wage_PKR" in predictions:
        print("\n💰 Minimum Wage Forecast:")
        print(f"Minimum Wage (PKR): {predictions['Minimum_Wage_PKR']}")

    # -----------------------------
    # 6. LLM Explanation (No PromptTemplate)
    # -----------------------------
    if explain:
        print("\n🧠 Generating AI Explanation...\n")

        formatted_data = "\n".join(
            [f"{k}: {v}" for k, v in predictions.items()]
        )

        # ✅ Direct prompt string (NO PromptTemplate)
        final_prompt = f"""
You are an expert economist.

Analyze the following predicted economic indicators for Pakistan in {target_year}:

{formatted_data}

Explain:
- What each indicator means
- Why it might increase or decrease
- Economic impact on people
- Relationship between inflation, wages, petrol, and currency

Give a clear and detailed explanation.
"""

        response = llm.invoke(final_prompt)

        print("📊 AI Economic Analysis:\n")
        print(response.content)

    return predictions


# -----------------------------
# 7. USER INPUT
# -----------------------------
try:
    user_year = int(input("Enter year for prediction (e.g., 2028): "))
    
    explain_choice = input("Do you want detailed AI explanation? (yes/no): ").lower()
    
    if explain_choice == "yes":
        results = predict_economy(user_year, explain=True)
    else:
        results = predict_economy(user_year, explain=False)

except ValueError:
    print("❌ Please enter a valid year!")

C:\Users\AB TRADERS\AppData\Local\Temp\ipykernel_17700\1665068437.py:50: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3:latest")



--- Economic Forecast for 2029 ---
--------------------------------------------------
Inflation Yoy                 : 45.75
Oil Price Usd Barrel          : 111.61
Exchange Rate Pkr Usd         : 394.4
Interest Rate                 : 25.99
Money Supply M2 Billion       : 44985.78
Pkr To Onedollar              : 387.67
Gold Price In Dolars          : 249.39
Petrol Price                  : 378.88
Minimum Wage Pkr              : 44263.48

💰 Minimum Wage Forecast:
Minimum Wage (PKR): 44263.48
